In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

# MCS-Ergebnisse (10.000 Pfade) — Median-Endkapital + Depletion Rate mit 95%-Wilson-CI
data = {
    "Aggressive": {
        "Buy & Hold":  {"median": 245081.83, "dr": 4.58, "ci_lo": 4.19, "ci_hi": 5.01},
        "MSM":         {"median": 190664.14, "dr": 2.25, "ci_lo": 1.98, "ci_hi": 2.56},
        "HMM":         {"median": 192216.25, "dr": 0.53, "ci_lo": 0.41, "ci_hi": 0.69},
        "LSTM":        {"median": 248069.01, "dr": 3.73, "ci_lo": 3.38, "ci_hi": 4.12},
        "Transformer": {"median": 223872.59, "dr": 4.06, "ci_lo": 3.69, "ci_hi": 4.46},
    },
    "Low_Capital": {
        "Buy & Hold":  {"median": 213677.49, "dr": 0.62, "ci_lo": 0.48, "ci_hi": 0.79},
        "MSM":         {"median": 175778.42, "dr": 0.05, "ci_lo": 0.02, "ci_hi": 0.12},
        "HMM":         {"median": 176790.63, "dr": 0.00, "ci_lo": 0.00, "ci_hi": 0.04},
        "LSTM":        {"median": 218280.88, "dr": 0.27, "ci_lo": 0.19, "ci_hi": 0.39},
        "Transformer": {"median": 196158.02, "dr": 0.38, "ci_lo": 0.28, "ci_hi": 0.52},
    },
}

COLORS = {
    "Buy & Hold":  "#7f7f7f",   # Benchmark — neutral grau
    "MSM":         "#1f77b4",   # Markov — blau
    "HMM":         "#1f77b4",
    "LSTM":        "#d62728",   # Deep Learning — rot
    "Transformer": "#d62728",
}
MARKERS = {
    "Buy & Hold":  "s",
    "MSM":         "o",
    "HMM":         "D",
    "LSTM":        "^",
    "Transformer": "v",
}
LABEL_OFFSET = {
    "Buy & Hold":  (10, -4),
    "MSM":         (10, 4),
    "HMM":         (10, 4),
    "LSTM":        (10, 4),
    "Transformer": (10, -12),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (scenario, models) in zip(axes, data.items()):
    bh_x, bh_y = models["Buy & Hold"]["median"], models["Buy & Hold"]["dr"]

    # Benchmark-Quadranten
    ax.axhline(bh_y, color="gray", lw=0.8, ls="--", alpha=0.55, zorder=1)
    ax.axvline(bh_x, color="gray", lw=0.8, ls="--", alpha=0.55, zorder=1)

    for name, v in models.items():
        yerr = [[v["dr"] - v["ci_lo"]], [v["ci_hi"] - v["dr"]]]
        ax.errorbar(
            v["median"], v["dr"], yerr=yerr,
            fmt=MARKERS[name], color=COLORS[name],
            markersize=11, markeredgecolor="black", markeredgewidth=0.6,
            elinewidth=1.2, capsize=3, alpha=0.92, zorder=3,
        )
        ax.annotate(
            name, (v["median"], v["dr"]),
            xytext=LABEL_OFFSET[name], textcoords="offset points",
            fontsize=9.5, fontweight="bold", zorder=4,
        )

    ax.set_xlabel("Median-Endkapital nach 25 Jahren", fontsize=10.5)
    ax.set_ylabel("Depletion Rate (%) — 95%-Wilson-CI", fontsize=10.5)
    ax.set_title(scenario.replace("_", "-") + "-Szenario",
                 fontsize=11.5, fontweight="bold")
    ax.grid(True, alpha=0.25, linestyle=":")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x/1000:.0f}k €"))

    # etwas Padding rechts/oben, damit Annotationen reinpassen
    xlo, xhi = ax.get_xlim()
    ylo, yhi = ax.get_ylim()
    ax.set_xlim(xlo, xhi + (xhi - xlo) * 0.10)
    ax.set_ylim(max(0, ylo - (yhi - ylo) * 0.05), yhi + (yhi - ylo) * 0.10)

legend_elements = [
    Patch(facecolor="#1f77b4", edgecolor="black", label="Markov-Modelle (HMM, MSM)"),
    Patch(facecolor="#d62728", edgecolor="black", label="DL-Modelle (LSTM, Transformer)"),
    Patch(facecolor="#7f7f7f", edgecolor="black", label="Benchmark (60/40 Buy & Hold)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=10)

fig.suptitle(
    "Risiko-Rendite-Positionierung der Modelle in den Stress-Szenarien",
    fontsize=12.5, fontweight="bold", y=1.00,
)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig("../../assets/risk_return_positioning.png", dpi=300, bbox_inches="tight")
plt.show()